# Meta MMS ASR Training for Twi - OPTIMIZED WITH MANIFESTS
## Using Enhanced Multi-Rate Data Processing Manifests

This optimized notebook uses the standardized manifests generated by the Enhanced Multi-Rate Data Processing Pipeline (`002_enhanced_multi_rate_data_processing.ipynb`).

### Key Changes:
- **Uses Standardized Manifests**: Loads pre-processed manifests from the enhanced data pipeline
- **Simplified Data Loading**: No need for custom dataset scanning logic
- **Consistent Sample Rates**: Uses 16kHz manifests optimized for ASR models
- **Quality-Filtered Data**: Pre-filtered for duration, text length, and audio quality
- **Multi-Dialect Support**: Automatically includes balanced dialect representation

In [ ]:
# Installation and imports
!pip install -q transformers datasets torch torchaudio evaluate jiwer tensorboard
!pip install -q accelerate librosa soundfile pandas

import os
import sys
import json
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Union
from dataclasses import dataclass
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Audio processing
import torchaudio
import librosa
import soundfile as sf

# MMS model imports
from transformers import (
    Wav2Vec2ForCTC,
    AutoProcessor,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from datasets import Dataset, DatasetDict, Audio
import evaluate

print(f"🔥 PyTorch version: {torch.__version__}")
print(f"🎯 CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"📊 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# Environment detection and path configuration
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("🔍 Detected Google Colab environment")
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_PATH = Path('/content/drive/MyDrive/voice-lab-data')
else:
    print("🔍 Detected local environment")
    DATA_PATH = Path('/opt/kasaai/apps/voice-lab/data')

print(f"📂 Data path: {DATA_PATH}")

# Configuration for manifest-based training
@dataclass
class ManifestBasedMMSConfig:
    """Configuration for MMS training using enhanced manifests."""
    # Model settings
    model_name: str = "facebook/mms-1b-all"
    primary_adapter: str = "aka"  # Akan (includes Twi)
    
    # Data paths
    data_dir: Path = DATA_PATH
    manifest_dir: Path = DATA_PATH / "processed/model_manifests"
    output_dir: Path = DATA_PATH / "models/mms_twi_manifest_optimized"
    cache_dir: Path = DATA_PATH / "cache"
    
    # Training parameters
    batch_size: int = 8
    gradient_accumulation_steps: int = 4
    learning_rate: float = 3e-4
    warmup_steps: int = 500
    num_train_epochs: int = 10
    fp16: bool = True
    gradient_checkpointing: bool = True
    
    # Audio settings (match enhanced processing)
    sample_rate: int = 16000
    
    # Evaluation
    eval_steps: int = 250
    save_steps: int = 250
    logging_steps: int = 50
    
    # MMS specific
    freeze_feature_encoder: bool = True
    mask_time_prob: float = 0.05
    layerdrop: float = 0.0
    
    def __post_init__(self):
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.cache_dir.mkdir(parents=True, exist_ok=True)

config = ManifestBasedMMSConfig()
print(f"📋 Manifest-based MMS Configuration loaded")
print(f"   Data directory: {config.data_dir}")
print(f"   Manifest directory: {config.manifest_dir}")
print(f"   Output directory: {config.output_dir}")

In [ ]:
# Enhanced manifest loader for MMS training
class EnhancedManifestLoader:
    """Load and process manifests from Enhanced Multi-Rate Data Processing Pipeline."""
    
    def __init__(self, config: ManifestBasedMMSConfig):
        self.config = config
        
    def load_mms_manifests(self) -> Dict[str, pd.DataFrame]:
        """Load MMS training manifests from enhanced processing pipeline."""
        print("📋 Loading MMS manifests from enhanced data processing...")
        
        manifests = {}
        
        # Look for MMS manifest files
        mms_manifest_files = {
            'train': self.config.manifest_dir / 'mms_train.csv',
            'val': self.config.manifest_dir / 'mms_val.csv'
        }
        
        # Fallback to other ASR manifests if MMS-specific ones don't exist
        fallback_manifests = {
            'train': [
                self.config.manifest_dir / 'faster_whisper_train.csv',
                self.config.manifest_dir / 'wav2vec2_train.csv'
            ],
            'val': [
                self.config.manifest_dir / 'faster_whisper_val.csv',
                self.config.manifest_dir / 'wav2vec2_val.csv'
            ]
        }
        
        for split_name in ['train', 'val']:
            manifest_path = mms_manifest_files[split_name]
            
            if manifest_path.exists():
                print(f"   ✅ Found MMS manifest: {manifest_path.name}")
                try:
                    df = pd.read_csv(manifest_path)
                    manifests[split_name] = df
                except Exception as e:
                    print(f"   ⚠️ Error loading {manifest_path}: {e}")
                    manifests[split_name] = pd.DataFrame()
            else:
                print(f"   ⚠️ MMS manifest not found: {manifest_path.name}")
                print(f"   🔄 Looking for fallback manifests...")
                
                # Try fallback manifests
                fallback_df = pd.DataFrame()
                for fallback_path in fallback_manifests[split_name]:
                    if fallback_path.exists():
                        print(f"   ✅ Using fallback: {fallback_path.name}")
                        try:
                            df = pd.read_csv(fallback_path)
                            fallback_df = pd.concat([fallback_df, df], ignore_index=True)
                        except Exception as e:
                            print(f"   ⚠️ Error loading {fallback_path}: {e}")
                            continue
                
                if not fallback_df.empty:
                    manifests[split_name] = fallback_df
                else:
                    print(f"   ❌ No valid manifests found for {split_name}")
                    manifests[split_name] = self._create_dummy_manifest()
        
        # Validate manifest format
        for split_name, df in manifests.items():
            if not df.empty:
                manifests[split_name] = self._validate_manifest_format(df, split_name)
        
        return manifests
    
    def _validate_manifest_format(self, df: pd.DataFrame, split_name: str) -> pd.DataFrame:
        """Validate and standardize manifest format."""
        print(f"   🔍 Validating {split_name} manifest format...")
        
        # Expected columns for ASR manifests
        required_columns = ['audio_path', 'transcript']
        optional_columns = ['duration', 'speaker_id', 'language', 'dataset']
        
        # Check required columns
        missing_required = set(required_columns) - set(df.columns)
        if missing_required:
            print(f"   ❌ Missing required columns: {missing_required}")
            print(f"   Available columns: {list(df.columns)}")
            
            # Try to map common column name variations
            column_mappings = {
                'text': 'transcript',
                'transcription': 'transcript',
                'sentence': 'transcript',
                'audio': 'audio_path',
                'path': 'audio_path',
                'file_path': 'audio_path'
            }
            
            for old_col, new_col in column_mappings.items():
                if old_col in df.columns and new_col in missing_required:
                    df = df.rename(columns={old_col: new_col})
                    print(f"   🔄 Mapped {old_col} → {new_col}")
        
        # Add missing optional columns with defaults
        for col in optional_columns:
            if col not in df.columns:
                if col == 'duration':
                    df[col] = 5.0  # Default duration
                elif col == 'speaker_id':
                    df[col] = 'unknown_speaker'
                elif col == 'language':
                    df[col] = 'tw'
                elif col == 'dataset':
                    df[col] = 'processed'
        
        # Filter valid entries
        initial_count = len(df)
        
        # Check if audio files exist
        if 'audio_path' in df.columns:
            df['audio_exists'] = df['audio_path'].apply(lambda x: Path(x).exists() if pd.notna(x) else False)
            df = df[df['audio_exists']].drop(columns=['audio_exists'])
        
        # Filter out empty transcripts
        if 'transcript' in df.columns:
            df = df[df['transcript'].str.len() > 0]
        
        final_count = len(df)
        retention_rate = (final_count / initial_count * 100) if initial_count > 0 else 0
        
        print(f"   📊 {split_name}: {final_count}/{initial_count} samples retained ({retention_rate:.1f}%)")
        
        return df
    
    def _create_dummy_manifest(self) -> pd.DataFrame:
        """Create dummy manifest for testing."""
        print("   📝 Creating dummy manifest for testing...")
        
        dummy_dir = self.config.data_dir / 'dummy_audio_mms'
        dummy_dir.mkdir(parents=True, exist_ok=True)
        
        samples = [
            "Wo ho te sɛn? Me ho yɛ.",
            "Me ma wo akye.",
            "Nante yie.",
            "Me da wo ase.",
            "Ɛyɛ dɛn?"
        ]
        
        dummy_data = []
        for i, text in enumerate(samples):
            audio_file = dummy_dir / f"dummy_mms_{i:03d}.wav"
            
            if not audio_file.exists():
                # Create dummy audio
                duration = np.random.uniform(2.0, 5.0)
                dummy_audio = np.random.normal(0, 0.1, int(16000 * duration)).astype(np.float32)
                sf.write(audio_file, dummy_audio, 16000)
            
            dummy_data.append({
                'audio_path': str(audio_file),
                'transcript': text,
                'duration': sf.info(str(audio_file)).duration,
                'speaker_id': f'dummy_speaker_{i % 2}',
                'language': 'tw',
                'dataset': 'dummy'
            })
        
        return pd.DataFrame(dummy_data)

# Initialize manifest loader
manifest_loader = EnhancedManifestLoader(config)
manifests = manifest_loader.load_mms_manifests()

print(f"\n📊 Manifest Loading Summary:")
for split_name, df in manifests.items():
    print(f"   {split_name}: {len(df)} samples")
    if not df.empty:
        print(f"     Speakers: {df['speaker_id'].nunique()}")
        print(f"     Duration: {df['duration'].sum()/60:.1f} minutes")
        print(f"     Sample audio path: {df['audio_path'].iloc[0]}")

In [ ]:
# Load MMS model with adapter support
class ManifestBasedMMSModel:
    """MMS model loader optimized for manifest-based training."""
    
    def __init__(self, config: ManifestBasedMMSConfig):
        self.config = config
        self.model = None
        self.processor = None
        
    def load_model_and_processor(self):
        """Load MMS model and processor with Twi adapter."""
        print(f"🚀 Loading MMS model: {self.config.model_name}")
        
        # Load processor
        self.processor = AutoProcessor.from_pretrained(
            self.config.model_name,
            cache_dir=self.config.cache_dir
        )
        
        # Set target language to Akan (Twi)
        try:
            self.processor.tokenizer.set_target_lang(self.config.primary_adapter)
            print(f"   ✅ Target language set to: {self.config.primary_adapter}")
        except Exception as e:
            print(f"   ⚠️ Could not set target language: {e}")
        
        # Load model
        self.model = Wav2Vec2ForCTC.from_pretrained(
            self.config.model_name,
            cache_dir=self.config.cache_dir,
            ignore_mismatched_sizes=True,
            mask_time_prob=self.config.mask_time_prob,
            layerdrop=self.config.layerdrop
        )
        
        # Load adapter
        try:
            self.model.load_adapter(self.config.primary_adapter)
            print(f"   ✅ Adapter loaded: {self.config.primary_adapter}")
        except Exception as e:
            print(f"   ⚠️ Could not load adapter: {e}")
        
        # Apply optimizations
        if self.config.freeze_feature_encoder:
            self.model.freeze_feature_encoder()
            print("   ❄️ Feature encoder frozen")
        
        if self.config.gradient_checkpointing:
            self.model.gradient_checkpointing_enable()
            print("   ✅ Gradient checkpointing enabled")
        
        print(f"\n📊 Model Summary:")
        print(f"   Parameters: {self.model.num_parameters():,}")
        print(f"   Vocabulary size: {len(self.processor.tokenizer)}")
        
        return self.model, self.processor

# Initialize model
mms_model_loader = ManifestBasedMMSModel(config)
model, processor = mms_model_loader.load_model_and_processor()

In [ ]:
# Convert manifests to HuggingFace datasets
class ManifestToDatasetConverter:
    """Convert manifest DataFrames to HuggingFace datasets."""
    
    def __init__(self, processor, config: ManifestBasedMMSConfig):
        self.processor = processor
        self.config = config
        
    def create_datasets(self, manifests: Dict[str, pd.DataFrame]) -> DatasetDict:
        """Convert manifests to HuggingFace datasets."""
        print("🔄 Converting manifests to HuggingFace datasets...")
        
        datasets = {}
        
        for split_name, df in manifests.items():
            if df.empty:
                print(f"   ⚠️ Empty {split_name} manifest, skipping")
                continue
                
            print(f"   📊 Processing {split_name}: {len(df)} samples")
            
            # Convert to HuggingFace dataset
            dataset = Dataset.from_pandas(df)
            
            # Add audio column
            dataset = dataset.cast_column("audio_path", Audio(sampling_rate=self.config.sample_rate))
            
            # Rename audio_path to audio for consistency
            dataset = dataset.rename_column("audio_path", "audio")
            
            datasets[split_name] = dataset
        
        # Ensure we have at least train and validation sets
        if 'train' not in datasets and 'val' in datasets:
            print("   🔄 Using val set as train set")
            datasets['train'] = datasets['val']
        
        if 'train' in datasets and 'val' not in datasets:
            print("   🔄 Splitting train set for validation")
            train_val_split = datasets['train'].train_test_split(test_size=0.1, seed=42)
            datasets['train'] = train_val_split['train']
            datasets['val'] = train_val_split['test']
        
        dataset_dict = DatasetDict(datasets)
        
        print(f"\n✅ Dataset conversion complete:")
        for split_name, dataset in dataset_dict.items():
            print(f"   {split_name}: {len(dataset)} samples")
        
        return dataset_dict
    
    def prepare_dataset(self, batch):
        """Prepare batch for training."""
        # Process audio
        audio_arrays = [x["array"] for x in batch["audio"]]
        
        inputs = self.processor(
            audio_arrays,
            sampling_rate=self.config.sample_rate,
            return_tensors="pt",
            padding=True,
            truncation=True
        )
        
        # Process text labels
        with self.processor.as_target_processor():
            labels = self.processor.tokenizer(
                batch["transcript"],
                return_tensors="pt",
                padding=True,
                truncation=True
            )
        
        return {
            "input_values": inputs.input_values,
            "labels": labels.input_ids
        }

# Convert manifests to datasets
if manifests['train'] is not None and not manifests['train'].empty:
    converter = ManifestToDatasetConverter(processor, config)
    datasets = converter.create_datasets(manifests)
    
    # Apply preprocessing
    print("\n🔄 Applying preprocessing...")
    processed_datasets = datasets.map(
        converter.prepare_dataset,
        batched=True,
        batch_size=8,
        remove_columns=datasets["train"].column_names,
        desc="Preprocessing"
    )
    
    print("✅ Preprocessing complete!")
else:
    print("❌ No valid training data found in manifests")
    processed_datasets = None

In [ ]:
# Data collator and training setup
from dataclasses import dataclass
from typing import Dict, List, Union
import torch

@dataclass
class DataCollatorCTCWithPadding:
    """Data collator for CTC training."""
    
    processor: AutoProcessor
    padding: Union[bool, str] = True
    
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # Separate inputs and labels
        input_features = [{"input_values": feature["input_values"]} for feature in features]
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        
        # Pad inputs
        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt",
        )
        
        # Pad labels
        with self.processor.as_target_processor():
            labels_batch = self.processor.pad(
                label_features,
                padding=self.padding,
                return_tensors="pt",
            )
        
        # Replace padding with -100 for loss computation
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        
        batch["labels"] = labels
        
        return batch

# Initialize data collator
data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)

print("✅ Data collator initialized")

In [ ]:
# Training setup and execution
def compute_metrics(eval_pred):
    """Compute WER and other metrics."""
    pred_logits = eval_pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)
    
    # Handle label padding
    eval_pred.label_ids[eval_pred.label_ids == -100] = processor.tokenizer.pad_token_id
    
    # Decode predictions and references
    pred_str = processor.batch_decode(pred_ids)
    label_str = processor.batch_decode(eval_pred.label_ids, group_tokens=False)
    
    # Compute metrics
    wer_metric = evaluate.load("wer")
    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    
    return {"wer": wer}

if processed_datasets is not None:
    print("🏋️ Setting up training...")
    
    training_args = TrainingArguments(
        output_dir=str(config.output_dir),
        group_by_length=True,
        per_device_train_batch_size=config.batch_size,
        per_device_eval_batch_size=config.batch_size,
        gradient_accumulation_steps=config.gradient_accumulation_steps,
        eval_strategy="steps",
        eval_steps=config.eval_steps,
        save_strategy="steps",
        save_steps=config.save_steps,
        logging_steps=config.logging_steps,
        learning_rate=config.learning_rate,
        num_train_epochs=config.num_train_epochs,
        warmup_steps=config.warmup_steps,
        fp16=config.fp16,
        gradient_checkpointing=config.gradient_checkpointing,
        save_total_limit=3,
        load_best_model_at_end=True,
        metric_for_best_model="wer",
        greater_is_better=False,
        push_to_hub=False,
        remove_unused_columns=False,
    )
    
    trainer = Trainer(
        model=model,
        data_collator=data_collator,
        args=training_args,
        compute_metrics=compute_metrics,
        train_dataset=processed_datasets["train"],
        eval_dataset=processed_datasets["val"] if "val" in processed_datasets else processed_datasets["train"],
        tokenizer=processor.feature_extractor,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    )
    
    print("✅ Training setup complete!")
    print(f"   Training samples: {len(processed_datasets['train'])}")
    print(f"   Validation samples: {len(processed_datasets.get('val', processed_datasets['train']))}")
    print(f"   Epochs: {config.num_train_epochs}")
    print(f"   Batch size: {config.batch_size}")
    print(f"   Learning rate: {config.learning_rate}")
    
    print("\n🚀 Ready to start training! Uncomment the next line to begin:")
    print("# train_result = trainer.train()")
    
    # Uncomment to start training
    # train_result = trainer.train()
    # trainer.save_model()
    # processor.save_pretrained(config.output_dir)
    # print(f"✅ Training complete! Model saved to {config.output_dir}")
    
else:
    print("❌ Cannot start training - no processed datasets available")

In [ ]:
# Evaluation and testing
def test_model_with_manifests():
    """Test the model using manifest data."""
    print("🧪 Testing model with manifest data...")
    
    if processed_datasets is None or 'val' not in processed_datasets:
        print("   ⚠️ No validation data available for testing")
        return
    
    # Get a few test samples
    test_samples = processed_datasets['val'].select(range(min(5, len(processed_datasets['val']))))
    
    model.eval()
    
    for i, sample in enumerate(test_samples):
        print(f"\n--- Sample {i+1} ---")
        
        # Get original text if available
        if 'val' in manifests and i < len(manifests['val']):
            expected_text = manifests['val'].iloc[i]['transcript']
            print(f"Expected: {expected_text}")
        
        # Prepare input
        input_values = sample['input_values'].unsqueeze(0)
        
        if torch.cuda.is_available():
            input_values = input_values.cuda()
        
        # Inference
        with torch.no_grad():
            logits = model(input_values).logits
        
        # Decode prediction
        predicted_ids = torch.argmax(logits, dim=-1)
        transcription = processor.batch_decode(predicted_ids)[0]
        
        print(f"Predicted: {transcription}")

def generate_usage_examples():
    """Generate usage examples for the trained model."""
    print("\n📚 Usage Examples:")
    
    usage_code = f'''
# Load the trained model
from transformers import Wav2Vec2ForCTC, AutoProcessor
import torch
import librosa

model_path = "{config.output_dir}"
model = Wav2Vec2ForCTC.from_pretrained(model_path)
processor = AutoProcessor.from_pretrained(model_path)

# Set target language
processor.tokenizer.set_target_lang("{config.primary_adapter}")
model.load_adapter("{config.primary_adapter}")

# Transcribe audio
audio, sr = librosa.load("twi_audio.wav", sr=16000)
inputs = processor(audio, sampling_rate=16000, return_tensors="pt", padding=True)

with torch.no_grad():
    logits = model(**inputs).logits

predicted_ids = torch.argmax(logits, dim=-1)
transcription = processor.batch_decode(predicted_ids)[0]
print(f"Transcription: {{transcription}}")
'''
    
    print(usage_code)

print("\n🎯 Testing and Usage Functions Ready:")
print("# test_model_with_manifests()")
print("# generate_usage_examples()")

## Summary: Manifest-Based MMS Training

This notebook demonstrates how to use the standardized manifests from the Enhanced Multi-Rate Data Processing Pipeline:

### ✅ Key Improvements:

1. **Standardized Data Loading**
   - Uses pre-processed manifests instead of custom scanning logic
   - Consistent format across all training notebooks
   - Quality-filtered data with proper audio normalization

2. **Simplified Pipeline**
   - Removed complex dataset discovery logic
   - Focus on model training rather than data preprocessing
   - Automatic fallback to compatible manifests

3. **Enhanced Reliability**
   - Robust error handling and validation
   - Fallback mechanisms for missing manifests
   - Consistent sample rates (16kHz for ASR)

4. **Better Integration**
   - Works seamlessly with the enhanced data processing pipeline
   - Consistent speaker identification across datasets
   - Balanced dialect representation

### 🔄 Migration Steps for Other Notebooks:

1. Replace custom dataset scanning with manifest loading
2. Use standardized column names (`audio_path`, `transcript`, etc.)
3. Remove duplicate audio preprocessing (already done in enhanced pipeline)
4. Add manifest validation and fallback logic
5. Update paths to use manifest directory structure

### 📊 Expected Benefits:

- **Faster Setup**: No need to scan raw directories
- **Consistent Quality**: Pre-filtered and normalized data
- **Better Performance**: Optimized sample rates per model type
- **Easier Maintenance**: Single source of truth for data processing